# modeling_v13 — 2차 선별 (Colab): **Conservative + GA**

M1 다이어트 `Conservative` 셋에 **유전 알고리즘(GA)** 2차 선별. GroupKFold(C20) OOF 를 적합도로 진화.
**고정** core10(10)+필수 5센서 champion(항상 포함, floor 보장), **가변** `Conservative` diet−고정만 GA가 선택.

### ▶ Colab 실행 순서
1. 아래 **셋업 셀**을 먼저 실행. 필요한 지원 파일 4개
   (`v13_select_colab.py`, `v13_fdc_pool_wf_oof.csv.gz`, `core10_meta_wf.csv`, `feature_diet_selected.json`)를
   자동 탐색하고, 못 찾으면 **업로드 창**이 떠서 그 자리에서 파일을 골라 올리면 된다.
2. 이후 셀을 순서대로 실행. **런타임 유형 = CPU** 로 충분(이 LightGBM 경로는 GPU 이득 없음).
3. 끝나면 `select_result_Conservative_GA.json` 생성 → 로컬로 내려받아 4개 비교에 사용.

> 두 GA 노트북(Conservative/Balanced)은 독립 → **Colab 세션 2개로 병렬** 실행 가능.

In [ ]:
# ── 셋업: 지원 파일 4개 자동 탐색 + (없으면) 업로드 창 ──
import os, json, warnings
warnings.filterwarnings("ignore")
import pandas as pd

need = ["v13_select_colab.py", "v13_fdc_pool_wf_oof.csv.gz",
        "core10_meta_wf.csv", "feature_diet_selected.json"]

def find_base(cands):
    for b in cands:
        if all(os.path.exists(os.path.join(b, f)) for f in need):
            return b
    return None

# 흔한 위치들 자동 탐색 (드래그 업로드/Drive/서브폴더 모두 커버)
CANDIDATES = [".", "/content", "/content/colab_GA",
              "/content/drive/MyDrive/colab_GA",
              "/content/drive/MyDrive/modeling_v13/colab_GA"]
BASE = find_base(CANDIDATES)

# 못 찾으면 Colab 업로드 창으로 직접 올리기 (여러 파일 한 번에 선택 가능)
if BASE is None:
    try:
        from google.colab import files
        print("지원 파일을 못 찾음 → 아래에서 4개 파일을 선택해 업로드하세요:")
        print(" ", ", ".join(need))
        files.upload()                 # /content(현재 폴더)로 업로드됨
        BASE = find_base(["."])
    except Exception:
        BASE = None

missing = [] if BASE else need
assert BASE is not None and not missing, (
    f"아직 누락: {need}\n"
    "→ 왼쪽 파일 패널에 4개를 올리거나, Drive 사용 시 BASE 를 그 폴더로 직접 지정하세요.")
print("모든 파일 확인 · BASE =", os.path.abspath(BASE))

In [ ]:
import sys; sys.path.insert(0, BASE)
import v13_select_colab as sc

PRESET = "Conservative"
df, y, groups, g = sc.load(PRESET, base=BASE)
ok, have = sc.floor_ok(g["fixed"])
print("df:", df.shape)
print(f"fixed(core10+champion)={len(g['fixed'])}  prunable(diet-fixed)={len(g['prunable'])}")
print("fixed floor:", ok, have)

## GA 선별

기본값 `pop=16, gens=8, fit_splits=3`. Colab 이 빠르면 `pop`·`gens` 를 키워 탐색을 넓혀도 됨.
적합도 평가마다 프록시 LGBM(200라운드) OOF 3-fold → fit 수가 많다(수백 회). 진행 로그가 세대별로 찍힌다.

In [ ]:
import time; t = time.time()
res = sc.ga_select(df, y, groups, g["fixed"], g["prunable"],
                   pop=16, gens=8, cx=0.6, mut=0.08, elite=2,
                   fit_splits=3, seed=42, verbose=True)
selected = res["best_subset"]
print(f"\nGA: prunable {len(g['prunable'])} -> {len(selected)}  best_GKF={res['best_rmse']:.3f}  evals={res['n_eval']}  ({time.time()-t:.0f}s)")

## 최종 OOF — 선택셋 vs baseline(2차선별 없음) · M8_PARAMS·705

In [ ]:
baseline = sc.final_report(df, y, groups, g["fixed"], g["prunable"], f"{PRESET}+diet(no-2nd)")
best     = sc.final_report(df, y, groups, g["fixed"], selected,       f"{PRESET}+GA")
tbl = pd.DataFrame([baseline, best])[["label","n_total","n_selected","KFold_OOF","GroupKFold_C20","floor_ok"]]
print(tbl.to_string(index=False)); tbl

## floor 재확인 · 저장

In [ ]:
okb, haveb = sc.floor_ok(best["features"]); assert okb, "FLOOR VIOLATION"
print("final floor:", okb, haveb)
out = {"preset": PRESET, "method": "GA",
       "baseline": {k: baseline[k] for k in ["n_total","KFold_OOF","GroupKFold_C20"]},
       "selected_result": {k: best[k] for k in ["n_total","n_selected","KFold_OOF","GroupKFold_C20","floor_ok"]},
       "selected_features": sorted(selected)}
fn = f"select_result_{PRESET}_GA.json"
json.dump(out, open(fn, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("saved:", fn, "-> 파일 패널에서 다운로드")

> **다음**: 이 `select_result_*.json` 과 RFECV 결과(로컬)를 모아 GroupKFold(C20) 기준 최적 조합 선택.
> 이후 그 셋을 고정하고 파라미터 재튜닝으로 절대 성능 확정.